# Ethnicity Metrics

Compares the distribution of perceived ethnicity between the ground-truth reference rankings (Semantic Scholar + BERT/ethnicolr cascade) and the LLM recommendations.

Distributions are pre-computed by `code/scripts/metrics/build_ethnicity_distributions.py` and read from `<results_dir>/ethnicity/distributions/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, '../..')

from libs.utils.config import get_results_path
from libs.visuals.constants import (
    ETHNICITY_PLOT_ORDER as ETH_ORDER,
    ETHNICITY_PLOT_COLOR_MAP as COLOR_MAP,
)

RESULTS = get_results_path()
DIST_DIR = RESULTS / 'ethnicity' / 'distributions'


In [ ]:
from libs.metrics.ethnicity_io import load_ethnicity_distributions

_dist = load_ethnicity_distributions(DIST_DIR, ETH_ORDER)
gt_overall   = _dist['gt_overall']
rec_overall  = _dist['rec_overall']
gt_field_df  = _dist['gt_per_field']
rec_field    = _dist['rec_per_field']
rec_model    = _dist['rec_per_model']

total_valid = rec_overall.sum()
print(f'GT unique researchers:        {gt_overall.sum():,}')
print(f'Valid LLM recommendations:    {total_valid:,}')


---
## 1. Ground Truth — Reference Rankings

In [ ]:
gt_pct = (gt_overall / gt_overall.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [COLOR_MAP.get(e, '#cccccc') for e in gt_overall.index]
gt_overall.plot(kind='bar', ax=ax, color=colors, width=0.6)
ax.set_title(f'Ground Truth — Ethnicity Distribution\n(n={gt_overall.sum():,} unique researchers)')
ax.set_xlabel('Perceived Ethnicity')
ax.set_ylabel('Researchers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, pct in zip(ax.patches, gt_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + gt_overall.sum()*0.005,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
gt_field_pct = gt_field_df.div(gt_field_df.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(gt_field_pct))
for eth in gt_field_pct.columns:
    vals = gt_field_pct[eth].values
    ax.bar(gt_field_pct.index, vals, bottom=bottom,
           color=COLOR_MAP.get(eth, '#ccc'), label=eth, width=0.6)
    bottom += vals
ax.set_title('Ground Truth — Ethnicity Distribution per Field (% unique researchers)')
ax.set_ylabel('Percentage (%)')
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=8)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


---
## 2. LLM Recommendations

Valid responses only (`cleaned`, `unchanged`).

In [ ]:
rec_pct = (rec_overall / rec_overall.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [COLOR_MAP.get(e, '#cccccc') for e in rec_overall.index]
rec_overall.plot(kind='bar', ax=ax, color=colors, width=0.6)
ax.set_title(f'Recommendations — Ethnicity Distribution\n(n={total_valid:,} valid responses)')
ax.set_xlabel('Perceived Ethnicity')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, pct in zip(ax.patches, rec_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total_valid*0.005,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
rec_field_pct = rec_field.div(rec_field.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(rec_field_pct))
for eth in rec_field_pct.columns:
    vals = rec_field_pct[eth].values
    ax.bar(rec_field_pct.index, vals, bottom=bottom,
           color=COLOR_MAP.get(eth, '#ccc'), label=eth, width=0.6)
    bottom += vals
ax.set_title('Recommendations — Ethnicity per Field (% valid responses)')
ax.set_ylabel('Percentage (%)')
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=8)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
rec_model_pct = rec_model.div(rec_model.sum(axis=1), axis=0) * 100
rec_model_pct = rec_model_pct.sort_values('White', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
left = np.zeros(len(rec_model_pct))
for eth in rec_model_pct.columns:
    vals = rec_model_pct[eth].values
    ax.barh(rec_model_pct.index, vals, left=left,
            color=COLOR_MAP.get(eth, '#ccc'), label=eth, height=0.7)
    left += vals
ax.set_title('Recommendations — Ethnicity per Model (% valid responses)')
ax.set_xlabel('Percentage (%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


---
## 3. Ground Truth vs. Recommendations — Side-by-side

`Unknown` is excluded from the comparison.

In [ ]:
KNOWN_ETH = [e for e in ETH_ORDER if e != 'Unknown']

gt_known  = gt_overall.reindex(KNOWN_ETH).fillna(0)
rec_known = rec_overall.reindex(KNOWN_ETH).fillna(0)

gt_known_pct  = gt_known  / gt_known.sum()  * 100
rec_known_pct = rec_known / rec_known.sum() * 100

x = np.arange(len(KNOWN_ETH))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_gt  = ax.bar(x - w/2, gt_known_pct.values,  w, label='Ground Truth',    color='#4878CF', alpha=0.85)
bars_rec = ax.bar(x + w/2, rec_known_pct.values, w, label='Recommendations', color='#D65F5F', alpha=0.85)
for bar in list(bars_gt) + list(bars_rec):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_title('Ground Truth vs. Recommendations — Ethnicity (%, excl. Unknown)')
ax.set_ylabel('Percentage (%)')
ax.set_xticks(x)
ax.set_xticklabels(KNOWN_ETH, rotation=15, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

delta = (rec_known_pct - gt_known_pct).round(2)
compare_df = pd.DataFrame({'GT (%)': gt_known_pct.round(2),
                          'Rec (%)': rec_known_pct.round(2),
                          'Delta (pp)': delta})
compare_df.index.name = 'perceived_ethnicity'
print(compare_df.to_string())


In [ ]:
fields = sorted(gt_field_df.index)
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey=True)
axes = axes.flatten()

for ax, field in zip(axes, fields):
    gt_f  = gt_field_df.loc[field, KNOWN_ETH] if field in gt_field_df.index else pd.Series(dtype=float)
    rec_f = rec_field.loc[field, KNOWN_ETH]   if field in rec_field.index   else pd.Series(dtype=float)
    gt_f_pct  = (gt_f  / gt_f.sum()  * 100).fillna(0)
    rec_f_pct = (rec_f / rec_f.sum() * 100).fillna(0)
    x = np.arange(len(KNOWN_ETH))
    ax.bar(x - 0.2, gt_f_pct.values,  0.38, label='GT',  color='#4878CF', alpha=0.85)
    ax.bar(x + 0.2, rec_f_pct.values, 0.38, label='Rec', color='#D65F5F', alpha=0.85)
    ax.set_title(field, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([e.split()[0] for e in KNOWN_ETH], rotation=25, ha='right', fontsize=8)
    ax.set_ylabel('%')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
